In [1]:
# 2) Importação das bibliotecas

#importações python para manipulação dos dados
import os
import pandas as pd
from pandasql import sqldf
from dotenv import load_dotenv
import shutil

#importações para criação do agente, mensagens e ferramentas estruturadas
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain.memory import ConversationBufferWindowMemory
from langchain_community.callbacks import get_openai_callback #controle de tokens detalhado


#importações para RAG
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
#from langchain_chroma import Chroma
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import DirectoryLoader, PyPDFLoader


load_dotenv()
print("DEBUG: bibliotecas importadas com sucesso")



DEBUG: bibliotecas importadas com sucesso


In [2]:
#3) Mapeamento das chaves
chave = os.getenv("api_key") 

# Criar a pasta 'docs' e atribuir o endereço à variável file_path
docs_folder = "bulas"
if not os.path.exists(docs_folder):
    os.makedirs(docs_folder)
    print(f"Pasta '{docs_folder}' criada.")

# Criar ou carregar o banco de dados vetorial (ChromaDB)
chroma_db_path = "./chroma_db_bulas"

print(f"DEBUG: criada a pasta {docs_folder} e criado o caminho do chroma_db_path ")

DEBUG: criada a pasta bulas e criado o caminho do chroma_db_path 


In [3]:
#4) Configuração do RAG


# I. Carregar TODOS os Documentos (PDFs)
# Usando o DirectoryLoader para iterar sobre todos os arquivos .pdf na pasta
print(f"DEBUG: Iniciando carregamento de PDFs na pasta '{docs_folder}'...")
try:
    # O 'glob="*.pdf"' garante que apenas arquivos PDF sejam carregados.
    # Usa o PyPDFLoader para fazer a extração de texto de cada PDF.
    loader = DirectoryLoader(
        path=docs_folder,
        glob="*.pdf",
        loader_cls=PyPDFLoader,
        silent_errors=True # Para ignorar arquivos que não puderem ser lidos
    )
    documents = loader.load()
    print(f"DEBUG: Documentos carregados. Número total de documentos (arquivos): {len(documents)}")
except Exception as e:
    print(f"ERRO: Não foi possível carregar os documentos PDF. Erro: {e}")
    documents = []

# Adição de Verificação: Se não houver documentos, para o processo
if not documents:
    print("AVISO: NENHUM documento PDF encontrado ou carregado. O processo será encerrado.")
    exit() # Interrompe a execução se não houver documentos


#II. Dividir documentos em chunks (pedaços menores)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)

# DEBUG: Verifique se 'splits' não está vazio
if not splits:
    print("AVISO: A lista 'splits' está vazia! Nenhuma parte do documento foi processada. Verifique o conteúdo do arquivo e o chunk_size.")
else:
    print(f"DEBUG: Número de chunks criados: {len(splits)}")
    print(f"DEBUG: Primeiro chunk: {splits[0].page_content[:100]}...")

# Gerar embeddings
embeddings = OpenAIEmbeddings(api_key=chave)

# DEBUG: Teste se o gerador de embeddings funciona
try:
    test_embedding = embeddings.embed_query("olá mundo")
    print(f"DEBUG: Embedding de teste gerado com sucesso. Tamanho: {len(test_embedding)}")
except Exception as e:
    print(f"ERRO: Falha ao gerar embedding de teste. Isso pode indicar um problema com a API key, créditos ou conexão: {e}")
    # Se os embeddings falharem aqui, o vectorstore.from_documents também falhará.


vectorstore = None
if os.path.exists(chroma_db_path) and os.listdir(chroma_db_path):
    print(f"DEBUG: Carregando ChromaDB existente de '{chroma_db_path}'.")
    try:
        vectorstore = Chroma(persist_directory=chroma_db_path, embedding_function=embeddings)
    except Exception as e:
        print(f"ERRO: Falha ao carregar ChromaDB existente. Pode estar corrompido. Tentando recriar. Erro: {e}")
        import shutil
        shutil.rmtree(chroma_db_path, ignore_errors=True) # Remove e tenta recriar

if vectorstore is None: # Se não carregou ou falhou, tenta criar
    if splits:
        print(f"DEBUG: Criando novo ChromaDB em '{chroma_db_path}'.")
        try:
            vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings, persist_directory=chroma_db_path)
            print("DEBUG: ChromaDB criado com sucesso.")
        except Exception as e:
            print(f"ERRO: Falha crítica ao criar ChromaDB a partir de documentos. Certifique-se de que 'splits' não está vazio e os embeddings funcionam. Erro: {e}")
            vectorstore = None
    else:
        print("AVISO: Não foi possível criar ChromaDB, pois 'splits' está vazio.")



DEBUG: Iniciando carregamento de PDFs na pasta 'bulas'...


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)


DEBUG: Documentos carregados. Número total de documentos (arquivos): 956
DEBUG: Número de chunks criados: 5318
DEBUG: Primeiro chunk: ZOLFEST SPRAY
hemitartarato de zolpidem
APRESENTAÇÃO
Solução spray de 25 mg/ml: embalagem conten-
do...
DEBUG: Embedding de teste gerado com sucesso. Tamanho: 1536
DEBUG: Criando novo ChromaDB em './chroma_db_bulas'.
DEBUG: ChromaDB criado com sucesso.


In [4]:
#5) Criação de funções & Ferramentas
@tool
def buscar_documentos(query: str) -> str:
        """
        Busca informações relevantes na base de conhecimento do dataframe.
        Traz características relevantes sobre o detalhamento de cada coluna, como elas foram geradas e outros pontos de destaque.
        """
        retriever = vectorstore.as_retriever()
        docs = retriever.invoke(query)
        if docs:
            return "\n\n".join([doc.page_content for doc in docs])
        else:
            return "Não encontrei informações relevantes nos documentos para esta consulta."
print("DEBUG: ferramenta 'buscar_documentos' criada com sucesso")

DEBUG: ferramenta 'buscar_documentos' criada com sucesso


In [5]:
#6)Criação da memória dos agentes
main_memory = ConversationBufferWindowMemory(
    memory_key="chat_history",
    return_messages=True,
    k=5 # Mantenha as últimas 5 interações (5 pares de HumanMessage/AIMessage)
)
print("DEBUG: criada memória do renome_agent com sucesso")

DEBUG: criada memória do renome_agent com sucesso


C:\Users\Pichau\AppData\Local\Temp\ipykernel_70040\4058121605.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  main_memory = ConversationBufferWindowMemory(


In [6]:
# 7) Configuração do LLM

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.25,
    api_key=chave,
    max_tokens=800
)




# 8) Criação do agente
tools = [buscar_documentos]

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", """você é um auxiliar prestativo que faz a leitura de bulas de medicamentos quando solicitado para sanar dúvidas.
        sua função é:
        ->responder ao consumidor
        ->pesquisar informações com precisão nos documentos de bulas
        ->sempre garantir que a informação passada esteja no documento e nunca explicar nada fora do documentado
        ->responder condialmente
        
        sua única ferramenta é:
        
        ->buscar_documentos [utilize esta ferramenta para entender as caracteristicas da bula e pesquisar as perguntas que o usuário faz.]
        

        
        """),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

agent = create_openai_tools_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=main_memory,
    verbose=False,
    handle_parsing_errors=True
)

print("DEBUG: agente configurado com sucesso")

DEBUG: agente configurado com sucesso


In [7]:
# 9) Loop para interagir com o agente
chat_history = []

while True:
    user_input = input("Digite sua pergunta ou 'sair' para encerrar: ")
    if user_input.lower() == 'sair':
        print("Encerrando o agente. Até mais!")
        break

    with get_openai_callback() as cb:
        try:
            response = agent_executor.invoke({"input": user_input, "chat_history": chat_history})
            print(f"Resposta: {response['output']}\n")
            chat_history.extend([HumanMessage(content=user_input), AIMessage(content=response["output"])])
        except Exception as e:
            print(f"Ocorreu um erro durante a execução: {e}")
            continue



Resposta: A posologia do Motore é a seguinte:

- **Via de administração**: Motore deve ser ingerido por via oral, com um pouco de água.
- **Dose habitual para adultos**: 2 cápsulas a cada 12 horas, totalizando 500 mg de medicação a cada tomada, ou seja, duas tomadas diárias.
- **Duração do tratamento**: O tempo de tratamento dependerá da gravidade dos sintomas e da evolução da doença, não havendo restrições específicas para o uso prolongado. O tempo de uso ficará a critério do profissional de saúde.
- **Orientações adicionais**: Caso não ocorra a obtenção do efeito desejado, as doses não devem ser aumentadas além da dose preconizada, sendo recomendada orientação médica. Este medicamento é indicado para uso em adultos. É importante seguir a orientação do médico, respeitando sempre os horários, as doses e a duração do tratamento. Não interrompa o tratamento sem o conhecimento do seu médico. Este medicamento não deve ser partido, aberto ou mastigado.

Se precisar de mais informações, esto